# In-House vs Send-Out Optimization Model

This project simulates a laboratory test utilization model to support in-house versus send-out decisions. Using Python and pandas, I generated a synthetic lab dataset and calculated operational recommendations based on turnaround time (TAT), cost difference, and weekly test volume.

## Project goals
- Model test utilization patterns across selected laboratory assays
- Estimate relative send-out risk using a weighted score
- Export a clean dataset for Power BI dashboarding

## Approach
The model assigns a utilization score using weighted contributions from turnaround time, cost difference, and weekly volume to approximate operational decision-making.

## Imports

In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

## Test Catalog

Define laboratory tests with associated methodologies and turnaround time ranges.

In [2]:
test_catalog = [
    {
        "test_name": "Respiratory Pathogen Panel",
        "department": "Molecular",
        "methodology": "PCR",
        "tat_min": 2,
        "tat_max": 8
    },
    {
        "test_name": "Blood Culture",
        "department": "Microbiology",
        "methodology": "Culture",
        "tat_min": 24,
        "tat_max": 120
    },
    {
        "test_name": "C. difficile NAAT",
        "department": "Molecular",
        "methodology": "NAAT",
        "tat_min": 2,
        "tat_max": 10
    },
    {
        "test_name": "Fungal PCR Panel",
        "department": "Molecular",
        "methodology": "PCR",
        "tat_min": 48,
        "tat_max": 120
    }
]

## Data Generation

Simulate laboratory orders using randomized test selection, turnaround time, and order dates.

In [3]:
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

data = []

for i in range(100):
    test = random.choice(test_catalog)
    tat = round(random.uniform(test["tat_min"], test["tat_max"]), 1)

    delta = end_date - start_date
    random_date = start_date + timedelta(days=random.randint(0, delta.days))

    row = {
        "order_date": random_date,
        "test_name": test["test_name"],
        "department": test["department"],
        "methodology": test["methodology"],
        "tat_hours": tat
    }

    data.append(row)

## DataFrame Creation

Convert generated records into a structured pandas DataFrame.

In [4]:
df = pd.DataFrame(data)
df.head()

,order_date,test_name,department,methodology,tat_hours
0,2025-08-02,Respiratory Pathogen Panel,Molecular,PCR,6.9
1,2025-05-23,Fungal PCR Panel,Molecular,PCR,84.5
2,2025-04-07,Blood Culture,Microbiology,Culture,34.7
3,2025-04-16,Fungal PCR Panel,Molecular,PCR,94.2
4,2025-04-17,Respiratory Pathogen Panel,Molecular,PCR,3.4


## Feature Engineering

Create additional variables including send-out flags, cost estimates, and test volume.

In [5]:
df["send_out"] = df["tat_hours"] > 24

df["base_cost"] = df["test_name"].map({
    "Respiratory Pathogen Panel": 220,
    "C. difficile NAAT": 110,
    "Blood Culture": 55,
    "Fungal PCR Panel": 260
})

df["send_out_cost"] = df["base_cost"] * 1.8
df["cost_difference"] = df["send_out_cost"] - df["base_cost"]

In [6]:
def get_volume(test_name):
    if test_name == "Blood Culture":
        return random.randint(150, 300)
    elif test_name == "Respiratory Pathogen Panel":
        return random.randint(80, 150)
    elif test_name == "C. difficile NAAT":
        return random.randint(40, 100)
    else:  # Fungal PCR Panel
        return random.randint(5, 20)

df["weekly_volume"] = df["test_name"].apply(get_volume)

## Scoring Model

Calculate a weighted score based on cost, turnaround time, and volume to determine test utilization recommendations.

In [7]:
df["score"] = (
    (df["cost_difference"] * 0.4) +
    (df["tat_hours"] * 0.4) -
    (df["weekly_volume"] * 0.5)
)

## Recommendation Logic

Assign final recommendations based on score thresholds.

In [8]:
df["final_recommendation"] = "Keep In-House"
df.loc[df["score"] > 100, "final_recommendation"] = "Send Out"
df.loc[
    (df["score"] > 50) & (df["score"] <= 100),
    "final_recommendation"
] = "Consider In-House Expansion"

## Validation & Summary

Review dataset structure and validate distribution of recommendations.

In [9]:
df.head()

,order_date,test_name,department,methodology,tat_hours,send_out,base_cost,send_out_cost,cost_difference,weekly_volume,score,final_recommendation
0,2025-08-02,Respiratory Pathogen Panel,Molecular,PCR,6.9,False,220,396.0,176.0,103,21.66,Keep In-House
1,2025-05-23,Fungal PCR Panel,Molecular,PCR,84.5,True,260,468.0,208.0,13,110.50,Send Out
2,2025-04-07,Blood Culture,Microbiology,Culture,34.7,True,55,99.0,44.0,209,-73.02,Keep In-House
3,2025-04-16,Fungal PCR Panel,Molecular,PCR,94.2,True,260,468.0,208.0,19,111.38,Send Out
4,2025-04-17,Respiratory Pathogen Panel,Molecular,PCR,3.4,False,220,396.0,176.0,106,18.76,Keep In-House


In [10]:
df.groupby("test_name")[["tat_hours", "cost_difference", "weekly_volume", "score"]].mean().round(2)

,tat_hours,cost_difference,weekly_volume,score
test_name,,,,
Blood Culture,73.95,44.0,215.38,-60.51
C. difficile NAAT,6.17,88.0,67.46,3.94
Fungal PCR Panel,89.36,208.0,14.85,111.52
Respiratory Pathogen Panel,5.32,176.0,115.50,14.78


In [11]:
df["final_recommendation"].value_counts()

final_recommendation
Keep In-House                  74
Send Out                       23
Consider In-House Expansion     3
Name: count, dtype: int64

## Export

Export final dataset for use in Power BI dashboard.

In [12]:
df_final = df[[
    "order_date",
    "test_name",
    "department",
    "methodology",
    "tat_hours",
    "weekly_volume",
    "base_cost",
    "send_out_cost",
    "cost_difference",
    "score",
    "final_recommendation"
]]

df_final.to_csv("lab_utilization_model_clean_export.csv", index=False)